# Stratified User Sampling

Build a reproducible, proportion-preserving dev sample from the full H&M dataset, plus a tiny **dummy** subset for smoke tests.

| Stage | Path | Description |
|-------|------|-------------|
| Input | `dataset/full/` | Full H&M CSVs (`articles`, `customers`, `transactions_train`) |
| Sample | `dataset/sample/` | ~1 000 stratified users → Parquet (Hive layout) |
| Dummy | `dataset/dummy/` | 5 random users from full data, 10 transactions total → Parquet (Hive layout) |

**Skipped:** `dataset/full/images/`

### Global temporal split

| Role | Rule | Default (`cutoff = 2020-03-31`) |
|------|------|-----------------------------------|
| **Features** | `t_dat <= cutoff` | History through 2020-03-31 |
| **Stratification** | `t_dat <= cutoff` | Tier/recency from pre-cutoff history |
| **Model labels** | `t_dat > cutoff`, 7-day window | `(2020-03-31, 2020-04-07]` — prediction starts **2020-04-01** |

Sampled output keeps **all dates** for selected users so ranker training can build post-cutoff labels.

Runs on **local PySpark** (`local[*]`, single machine). Same code can later move to Glue with path config only.

Column types follow `docs/system-design/schema-info.md` (explicit casts; no `inferSchema`).

**Kernel:** `Fashion Reco (notebooks)` — see repo `README.md` for env setup.

Spec: `docs/superpowers/specs/2026-06-04-stratified-user-sampling-design.md`

## Configuration

Paths, sampling targets, and random seeds. Loaded from `notebooks/utils/config_loader.py`
(`load_sampling_config`) — override via environment variables for Glue or alternate runs.


In [1]:
import json
import os
import shutil
import sys
from datetime import datetime, timezone
from functools import reduce
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

for _nb in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (_nb / "utils").is_dir():
        sys.path[:0] = [str(_nb)]
        break

from utils.config_loader import is_glue_env, load_sampling_config

CONFIG = load_sampling_config()
IS_GLUE = is_glue_env()
CONFIG

{'input_path': '../dataset/full',
 'output_path': '../dataset/sample_2000_users',
 'dummy_output_path': '../dataset/dummy',
 'cutoff_date': '2020-03-31',
 'label_window_days': 7,
 'recency_days': 30,
 'target_n': 2000,
 'random_seed': 42,
 'target_tolerance': 50,
 'dummy_n_users': 5,
 'dummy_n_transactions': 10,
 'dummy_random_seed': 99}

## Spark Session

Local PySpark driver (or reuse the Glue-provided session) via `notebooks/utils/spark_session.py`.
Re-run safe: stale JVM connections are reset automatically.


In [2]:
from utils.spark_session import create_spark_session

spark = create_spark_session("stratified-user-sampling", is_glue=IS_GLUE)

## Pipeline Helpers

Schema casts, stratification labels, quota allocation, sampling, and Parquet writers. Shared by both the **sample** and **dummy** outputs.


In [3]:
# --- Stratified sampling pipeline (Glue-portable; no Pandas) ---
# Types per docs/system-design/schema-info.md — read CSV as strings, cast explicitly.


def cast_articles(df: DataFrame) -> DataFrame:
    """Cast articles columns per schema-info.md."""
    return df.select(
        F.col("article_id").cast("string"),
        F.col("product_code").cast("string"),
        F.col("prod_name").cast("string"),
        F.col("product_type_no").cast("int"),
        F.col("product_type_name").cast("string"),
        F.col("product_group_name").cast("string"),
        F.col("graphical_appearance_no").cast("int"),
        F.col("graphical_appearance_name").cast("string"),
        F.col("colour_group_code").cast("string"),
        F.col("colour_group_name").cast("string"),
        F.col("perceived_colour_value_id").cast("int"),
        F.col("perceived_colour_value_name").cast("string"),
        F.col("perceived_colour_master_id").cast("int"),
        F.col("perceived_colour_master_name").cast("string"),
        F.col("department_no").cast("int"),
        F.col("department_name").cast("string"),
        F.col("index_code").cast("string"),
        F.col("index_name").cast("string"),
        F.col("index_group_no").cast("int"),
        F.col("index_group_name").cast("string"),
        F.col("section_no").cast("int"),
        F.col("section_name").cast("string"),
        F.col("garment_group_no").cast("int"),
        F.col("garment_group_name").cast("string"),
        F.col("detail_desc").cast("string"),
    )


def cast_customers(df: DataFrame) -> DataFrame:
    """Cast customers columns per schema-info.md."""
    return df.select(
        F.col("customer_id").cast("string"),
        F.col("FN").cast("string"),
        F.col("Active").cast("string"),
        F.col("club_member_status").cast("string"),
        F.col("fashion_news_frequency").cast("string"),
        F.col("age").cast("int"),
        F.col("postal_code").cast("string"),
    )


def cast_transactions(df: DataFrame) -> DataFrame:
    """Cast transactions columns per schema-info.md."""
    return df.select(
        F.to_date("t_dat").alias("t_dat"),
        F.col("customer_id").cast("string"),
        F.col("article_id").cast("string"),
        F.col("price").cast("double"),
        F.col("sales_channel_id").cast("int"),
    )


def load_hm_csvs(
    spark: SparkSession, input_path: str
) -> tuple[DataFrame, DataFrame, DataFrame]:
    """Read raw H&M CSVs and apply schema casts (no inferSchema)."""
    read_csv = spark.read.option("header", True).option("inferSchema", False)
    return (
        cast_articles(read_csv.csv(f"{input_path}/articles.csv")),
        cast_customers(read_csv.csv(f"{input_path}/customers.csv")),
        cast_transactions(read_csv.csv(f"{input_path}/transactions_train.csv")),
    )


def add_transaction_partitions(df: DataFrame) -> DataFrame:
    """Hive partition keys for transactions (year/month per v1-requirements)."""
    return df.withColumn("year", F.date_format("t_dat", "yyyy")).withColumn(
        "month", F.date_format("t_dat", "MM")
    )


def write_parquet_hive(
    df: DataFrame,
    output_dir: Path,
    partition_cols: list[str] | None = None,
) -> None:
    """Write a DataFrame as Parquet with optional Hive-style partitioning."""
    output_path = output_dir.resolve()
    if output_path.exists():
        shutil.rmtree(output_path)

    writer = df.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(str(output_path))


def build_user_labels(
    customers_df: DataFrame,
    transactions_df: DataFrame,
    cutoff_date: str,
    recency_days: int,
) -> DataFrame:
    """Assign stratification labels to every customer using pre-cutoff history only.

    Labels drive proportional sampling. Post-cutoff transactions are ignored here
    (they still appear in the final filtered output after sampling).

    Parameters
    ----------
    customers_df : DataFrame
        H&M customers table. Required column: ``customer_id``.
    transactions_df : DataFrame
        H&M transactions. Required columns: ``customer_id``, ``t_dat`` (date).
    cutoff_date : str
        ISO date string (e.g. ``"2020-03-31"``). Only rows with ``t_dat <= cutoff``
        count toward tier and recency.
    recency_days : int
        Length of the active window immediately before cutoff. Active purchases
        are those with ``t_dat`` in ``(cutoff - recency_days, cutoff]``.

    Returns
    -------
    DataFrame
        One row per customer with columns:
        ``customer_id``, ``purchase_count_pre_cutoff``, ``last_purchase_pre_cutoff``,
        ``recent_purchase_count``, ``purchase_tier``, ``recency``.

        ``purchase_tier`` ∈ {new, cold, light, medium, heavy}.
        ``recency`` ∈ {n/a, active-recent, dormant} (n/a only for new tier).

    Notes
    -----
    Customers with no pre-cutoff transactions get tier ``new`` and recency ``n/a``.
    """
    cutoff = F.lit(cutoff_date).cast("date")
    # Exclusive lower bound: purchases on cutoff itself count as recent.
    recency_start = F.date_sub(cutoff, recency_days)

    # Segment labels must not leak future (post-cutoff) behavior.
    pre_cutoff_txns = transactions_df.filter(F.col("t_dat") <= cutoff)

    user_stats = pre_cutoff_txns.groupBy("customer_id").agg(
        F.count("*").alias("purchase_count_pre_cutoff"),
        F.max("t_dat").alias("last_purchase_pre_cutoff"),
        # Count purchases inside the recency window (open on left, closed on right).
        F.sum(
            F.when(
                (F.col("t_dat") > recency_start) & (F.col("t_dat") <= cutoff),
                1,
            ).otherwise(0)
        ).alias("recent_purchase_count"),
    )

    # Left join: customers with zero pre-cutoff txns stay in the pool as "new".
    labeled = customers_df.select("customer_id").join(
        user_stats, on="customer_id", how="left"
    )

    labeled = labeled.fillna(
        {"purchase_count_pre_cutoff": 0, "recent_purchase_count": 0}
    )

    # Tier boundaries match design spec (medium 6–19, heavy 20+).
    labeled = labeled.withColumn(
        "purchase_tier",
        F.when(F.col("purchase_count_pre_cutoff") == 0, F.lit("new"))
        .when(F.col("purchase_count_pre_cutoff") <= 2, F.lit("cold"))
        .when(F.col("purchase_count_pre_cutoff") <= 5, F.lit("light"))
        .when(F.col("purchase_count_pre_cutoff") <= 19, F.lit("medium"))
        .otherwise(F.lit("heavy")),
    )

    # Recency is only meaningful for buyers; new users share one sampling cell.
    labeled = labeled.withColumn(
        "recency",
        F.when(F.col("purchase_tier") == "new", F.lit("n/a"))
        .when(F.col("recent_purchase_count") >= 1, F.lit("active-recent"))
        .otherwise(F.lit("dormant")),
    )

    return labeled


def compute_quotas(labeled_df: DataFrame, target_n: int) -> list[dict]:
    """Allocate per-cell sample sizes so global mix matches population proportions.

    Uses the largest-remainder method so integer quotas sum exactly to ``target_n``,
    then caps each cell at its population size (small cells may yield fewer users).

    Parameters
    ----------
    labeled_df : DataFrame
        Output of ``build_user_labels``. Must include ``purchase_tier`` and
        ``recency``.
    target_n : int
        Desired number of distinct users to sample (e.g. 1000).

    Returns
    -------
    list[dict]
        One dict per stratification cell with keys:
        ``purchase_tier``, ``recency``, ``population``, ``quota``.
        Sum of ``quota`` may be less than ``target_n`` if many cells are tiny.

    Raises
    ------
    ValueError
        If ``labeled_df`` has no rows.
    """
    cell_counts = labeled_df.groupBy("purchase_tier", "recency").count().collect()
    total = sum(row["count"] for row in cell_counts)
    if total == 0:
        raise ValueError("No labeled users found")

    cells = []
    for row in cell_counts:
        # Proportional share before rounding (Hamilton / largest remainder).
        raw_quota = target_n * row["count"] / total
        floor_quota = int(raw_quota)
        cells.append(
            {
                "purchase_tier": row["purchase_tier"],
                "recency": row["recency"],
                "population": row["count"],
                "floor_quota": floor_quota,
                "remainder": raw_quota - floor_quota,
            }
        )

    # Distribute leftover slots to cells with largest fractional parts.
    assigned = sum(cell["floor_quota"] for cell in cells)
    remaining = target_n - assigned
    cells.sort(key=lambda cell: cell["remainder"], reverse=True)
    for idx in range(remaining):
        cells[idx % len(cells)]["floor_quota"] += 1

    # Never request more users than exist in a cell (sparse strata).
    for cell in cells:
        cell["quota"] = min(cell["floor_quota"], cell["population"])
        cell.pop("floor_quota", None)
        cell.pop("remainder", None)

    return cells


def sample_users(
    labeled_df: DataFrame, quotas: list[dict], random_seed: int
) -> DataFrame:
    """Draw a reproducible stratified subset of customer IDs.

    Within each cell, rows are ordered by ``rand(seed)`` then ``customer_id`` so
    ties break deterministically. The same seed and quotas yield the same sample.

    Parameters
    ----------
    labeled_df : DataFrame
        Labeled customers from ``build_user_labels``.
    quotas : list[dict]
        Per-cell targets from ``compute_quotas`` (``purchase_tier``, ``recency``,
        ``quota``).
    random_seed : int
        Fixed seed passed to ``F.rand`` for reproducible shuffles.

    Returns
    -------
    DataFrame
        Single column ``customer_id``, one row per sampled user (distinct).

    Raises
    ------
    ValueError
        If every cell has ``quota <= 0``.
    """
    sampled_parts: list[DataFrame] = []
    for cell in quotas:
        quota = cell["quota"]
        if quota <= 0:
            continue

        cell_df = labeled_df.filter(
            (F.col("purchase_tier") == cell["purchase_tier"])
            & (F.col("recency") == cell["recency"])
        )
        # limit() after orderBy() takes the top-k rows per stratum.
        sampled = cell_df.orderBy(F.rand(random_seed), "customer_id").limit(quota)
        sampled_parts.append(sampled.select("customer_id"))

    if not sampled_parts:
        raise ValueError("No users sampled — all cell quotas are zero")

    return reduce(DataFrame.unionByName, sampled_parts).distinct()


def filter_and_write(
    sampled_ids_df: DataFrame,
    customers_df: DataFrame,
    transactions_df: DataFrame,
    articles_df: DataFrame,
    output_path: str,
) -> dict[str, int]:
    """Subset star-schema tables to sampled users and write typed Parquet.

    Transactions are **not** re-filtered by cutoff: all dates for sampled users
    are kept so models can use full history. Articles are derived only from
    transactions that remain after the user filter.

    Output layout (Hive-style):
    - ``articles/`` — unpartitioned Parquet
    - ``customers/`` — unpartitioned Parquet
    - ``transactions/year=YYYY/month=MM/`` — partitioned by transaction month

    Parameters
    ----------
    sampled_ids_df : DataFrame
        Column ``customer_id`` — output of ``sample_users``.
    customers_df : DataFrame
        Full customers table.
    transactions_df : DataFrame
        Full transactions table (all ``t_dat`` values retained for sampled users).
    articles_df : DataFrame
        Full articles catalog.
    output_path : str
        Directory for ``articles/``, ``customers/``, ``transactions/`` Parquet tables.

    Returns
    -------
    dict[str, int]
        Row counts after filtering: ``users``, ``transactions``, ``articles``.
    """
    out = Path(output_path)
    out.mkdir(parents=True, exist_ok=True)

    sampled_ids = sampled_ids_df.select("customer_id").distinct()

    sampled_customers = customers_df.join(sampled_ids, on="customer_id", how="inner")
    sampled_transactions = transactions_df.join(sampled_ids, on="customer_id", how="inner")

    # Article dimension: only SKUs that appear in the sampled fact table.
    article_ids = sampled_transactions.select("article_id").distinct()
    sampled_articles = articles_df.join(article_ids, on="article_id", how="inner")

    write_parquet_hive(sampled_articles, out / "articles")
    write_parquet_hive(sampled_customers, out / "customers")
    write_parquet_hive(
        add_transaction_partitions(sampled_transactions),
        out / "transactions",
        partition_cols=["year", "month"],
    )

    return {
        "users": sampled_customers.count(),
        "transactions": sampled_transactions.count(),
        "articles": sampled_articles.count(),
    }

def build_dummy_from_full(
    customers_df: DataFrame,
    transactions_df: DataFrame,
    articles_df: DataFrame,
    dummy_path: str,
    n_users: int,
    n_transactions: int,
    random_seed: int,
    user_pool: DataFrame | None = None,
    cutoff_date: str = "2020-03-31",
    label_window_days: int = 7,
) -> dict[str, int | list[str]]:
    """Build a tiny smoke-test dataset directly from the full catalog.

    Picks ``n_users`` customer IDs at random (reproducible via ``random_seed``),
    then keeps at most ``n_transactions`` transactions among those users.
    Only customer rows for the sampled users and articles referenced by those
    transactions are written.

    Parameters
    ----------
    customers_df : DataFrame
        Full customers table.
    transactions_df : DataFrame
        Full transactions table.
    articles_df : DataFrame
        Full articles catalog.
    dummy_path : str
        Output root for the dummy dataset (same Hive layout as ``filter_and_write``).
    n_users : int
        Number of users to include (e.g. 5).
    n_transactions : int
        Maximum total transactions to keep across sampled users (e.g. 10).
    random_seed : int
        Seed for ``F.rand`` when drawing users and transactions.
    user_pool : DataFrame | None
        Optional single-column ``customer_id`` pool. When omitted, all customers
        are eligible.

    Returns
    -------
    dict[str, int | list[str]]
        Row counts plus the selected ``customer_id`` list.
    """
    pool = (
        user_pool.select("customer_id").distinct()
        if user_pool is not None
        else customers_df.select("customer_id")
    )

    label_start = F.to_date(F.lit(cutoff_date))
    label_end = F.date_add(label_start, label_window_days)
    users_with_label_window = transactions_df.filter(
        (F.col("t_dat") > label_start) & (F.col("t_dat") <= label_end)
    ).select("customer_id").distinct()
    eligible_pool = pool.join(users_with_label_window, on="customer_id", how="inner")
    if eligible_pool.limit(1).count() == 0:
        eligible_pool = pool

    dummy_ids = eligible_pool.orderBy(F.rand(random_seed), "customer_id").limit(n_users)
    selected_ids = [row.customer_id for row in dummy_ids.collect()]

    sampled_customers = customers_df.join(dummy_ids, on="customer_id", how="inner")
    user_transactions = transactions_df.join(dummy_ids, on="customer_id", how="inner")

    # Separate seed offset so user and transaction shuffles are independent.
    txn_seed = random_seed + 1
    label_window_txns = user_transactions.filter(
        (F.col("t_dat") > label_start) & (F.col("t_dat") <= label_end)
    )
    label_slots = max(1, min(n_transactions, 3))
    label_sample = label_window_txns.orderBy(
        F.rand(txn_seed), "customer_id", "t_dat", "article_id"
    ).limit(label_slots)
    remaining_slots = n_transactions - label_slots
    if remaining_slots > 0:
        other_sample = (
            user_transactions.join(
                label_sample, on=["customer_id", "t_dat", "article_id"], how="left_anti"
            )
            .orderBy(F.rand(txn_seed + 2), "customer_id", "t_dat", "article_id")
            .limit(remaining_slots)
        )
        sampled_transactions = label_sample.unionByName(other_sample)
    else:
        sampled_transactions = label_sample

    article_ids = sampled_transactions.select("article_id").distinct()
    sampled_articles = articles_df.join(article_ids, on="article_id", how="inner")

    out = Path(dummy_path)
    out.mkdir(parents=True, exist_ok=True)

    write_parquet_hive(sampled_articles, out / "articles")
    write_parquet_hive(sampled_customers, out / "customers")
    write_parquet_hive(
        add_transaction_partitions(sampled_transactions),
        out / "transactions",
        partition_cols=["year", "month"],
    )

    counts = {
        "users": sampled_customers.count(),
        "transactions": sampled_transactions.count(),
        "articles": sampled_articles.count(),
    }
    counts["customer_ids"] = selected_ids
    return counts


## Load dataset

In [4]:
input_path = CONFIG["input_path"]
for name in ("articles.csv", "customers.csv", "transactions_train.csv"):
    path = Path(input_path) / name
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")

# Load star-schema inputs (images/ intentionally omitted).
articles, customers, transactions = load_hm_csvs(spark, input_path)

## Run Stratified Sampling

Load full CSVs → label users → allocate quotas → draw sample → write `dataset/sample/` Parquet tables.


In [ ]:
# Pipeline: label → quota → sample → filter/write.
labeled_users = build_user_labels(
    customers,
    transactions,
    CONFIG["cutoff_date"],
    CONFIG["recency_days"],
)
labeled_users.cache()

quotas = compute_quotas(labeled_users, CONFIG["target_n"])
sampled_ids = sample_users(labeled_users, quotas, CONFIG["random_seed"])
sampled_ids.cache()

output_counts = filter_and_write(
    sampled_ids,
    customers,
    transactions,
    articles,
    CONFIG["output_path"],
)

print("Sampling complete:")
print(json.dumps({"quotas": quotas, "output_counts": output_counts}, indent=2))

Sampling complete:
{
  "quotas": [
    {
      "purchase_tier": "cold",
      "recency": "active-recent",
      "population": 14866,
      "quota": 11
    },
    {
      "purchase_tier": "light",
      "recency": "active-recent",
      "population": 17582,
      "quota": 13
    },
    {
      "purchase_tier": "new",
      "recency": "n/a",
      "population": 176666,
      "quota": 129
    },
    {
      "purchase_tier": "heavy",
      "recency": "dormant",
      "population": 227270,
      "quota": 166
    },
    {
      "purchase_tier": "light",
      "recency": "dormant",
      "population": 203764,
      "quota": 149
    },
    {
      "purchase_tier": "medium",
      "recency": "dormant",
      "population": 335351,
      "quota": 244
    },
    {
      "purchase_tier": "medium",
      "recency": "active-recent",
      "population": 51304,
      "quota": 37
    },
    {
      "purchase_tier": "cold",
      "recency": "dormant",
      "population": 222729,
      "quota": 162
    },

## Dummy dataset

Draw **5 random users** from the full dataset, keep **10 transactions** total among them, and write a tiny Parquet copy to `dataset/dummy/`. Same Hive layout as the sample — minimal footprint for pipeline smoke tests.


In [6]:
dummy_output_counts = build_dummy_from_full(
    customers,
    transactions,
    articles,
    dummy_path=CONFIG["dummy_output_path"],
    n_users=CONFIG["dummy_n_users"],
    n_transactions=CONFIG["dummy_n_transactions"],
    random_seed=CONFIG["dummy_random_seed"],
    cutoff_date=CONFIG["cutoff_date"],
    label_window_days=CONFIG["label_window_days"],
)

print("Dummy dataset complete:")
print(json.dumps(dummy_output_counts, indent=2))

Dummy dataset complete:
{
  "users": 5,
  "transactions": 10,
  "articles": 10,
  "customer_ids": [
    "aeb0430b6f1eb45079d047466435c30ff1394013eec26a9932a20a4053d05b81",
    "cd5606124c1e217911caab45aff99551058b98d63d671e9ebccadc28ed17e4c9",
    "66740961ead1151df6f22f2aadf96905e123d7ec1b510906de1fb0712aa2309a",
    "c6ea7392d73a4c4c1967300f71d67edd049b26b2a3d77696436091c3f52cd76b",
    "8fdb3b94e9dcbd55aaa3015ce5571277a88a4b1fcb5c08c1f2317ec63f740b17"
  ]
}


## Validate Sample Output

Quick integrity checks on the written Parquet: article coverage, segment proportions, and target-size tolerance.


In [ ]:
output_path = CONFIG["output_path"]
sample_articles = spark.read.parquet(f"{output_path}/articles")
sample_transactions = spark.read.parquet(f"{output_path}/transactions")

# article_ids in articles but not in transactions
transaction_article_ids = sample_transactions.select("article_id").distinct()
articles_not_in_transactions = (
    sample_articles.select("article_id").distinct()
    .join(transaction_article_ids, on="article_id", how="left_anti")
)

print(f"Total articles: {sample_articles.select('article_id').distinct().count()}")
print(f"Articles not in transactions: {articles_not_in_transactions.count()}")
# articles_not_in_transactions.orderBy("article_id").show(20, truncate=False)


Total articles: 13963


Articles not in transactions: 0
+----------+
|article_id|
+----------+
+----------+



In [ ]:
def proportion_table(df: DataFrame, group_cols: list[str], label: str) -> None:
    """Print a human-readable distribution table for QA (stdout only).

    Compares full vs sample segment mixes to confirm stratification worked.

    Parameters
    ----------
    df : DataFrame
        Labeled users or a subset (e.g. post-join with ``sampled_ids``).
    group_cols : list[str]
        Columns to group by (e.g. ``["purchase_tier"]`` or
        ``["purchase_tier", "recency"]``).
    label : str
        Title printed above the table (for notebook logs).

    Returns
    -------
    None
        Side effect: prints counts and percentages to stdout.
    """
    total = df.count()
    rows = (
        df.groupBy(*group_cols)
        .count()
        .orderBy(*group_cols)
        .collect()
    )
    print(f"\n{label} (n={total})")
    print(f"{' | '.join(group_cols)} | count | pct")
    print("-" * 60)
    for row in rows:
        key = ", ".join(str(row[col]) for col in group_cols)
        pct = 100.0 * row["count"] / total if total else 0.0
        print(f"{key} | {row['count']} | {pct:.2f}%")


# Join back to labeled rows to compare segment mix: population vs sample.
sampled_labeled = labeled_users.join(sampled_ids, on="customer_id", how="inner")

proportion_table(labeled_users, ["purchase_tier"], "Full dataset — purchase tier")
proportion_table(sampled_labeled, ["purchase_tier"], "Sample — purchase tier")

# Recency axis only applies to non-new tiers; exclude "new" for a fair comparison.
non_new_full = labeled_users.filter(F.col("purchase_tier") != "new")
non_new_sample = sampled_labeled.filter(F.col("purchase_tier") != "new")
proportion_table(
    non_new_full, ["purchase_tier", "recency"], "Full dataset — tier x recency (non-new)"
)
proportion_table(
    non_new_sample, ["purchase_tier", "recency"], "Sample — tier x recency (non-new)"
)

sampled_n = sampled_ids.count()
delta = abs(sampled_n - CONFIG["target_n"])
print(f"\nSampled users: {sampled_n} (target={CONFIG['target_n']}, delta={delta})")
print(
    f"Output rows — users: {output_counts['users']}, "
    f"transactions: {output_counts['transactions']}, "
    f"articles: {output_counts['articles']}"
)

# Hard checks before treating dataset/sample/ as ready for downstream pipelines.
assert output_counts["users"] == sampled_n
assert delta <= CONFIG["target_tolerance"], (
    f"Sampled {sampled_n} users, expected within {CONFIG['target_tolerance']} of "
    f"{CONFIG['target_n']}"
)
print("\nValidation passed.")


Full dataset — purchase tier (n=1371980)
purchase_tier | count | pct
------------------------------------------------------------
cold | 237595 | 17.32%
heavy | 349718 | 25.49%
light | 221346 | 16.13%
medium | 386655 | 28.18%
new | 176666 | 12.88%

Sample — purchase tier (n=1000)
purchase_tier | count | pct
------------------------------------------------------------
cold | 173 | 17.30%
heavy | 255 | 25.50%
light | 162 | 16.20%
medium | 281 | 28.10%
new | 129 | 12.90%

Full dataset — tier x recency (non-new) (n=1195314)
purchase_tier | recency | count | pct
------------------------------------------------------------
cold, active-recent | 14866 | 1.24%
cold, dormant | 222729 | 18.63%
heavy, active-recent | 122448 | 10.24%
heavy, dormant | 227270 | 19.01%
light, active-recent | 17582 | 1.47%
light, dormant | 203764 | 17.05%
medium, active-recent | 51304 | 4.29%
medium, dormant | 335351 | 28.06%

Sample — tier x recency (non-new) (n=871)
purchase_tier | recency | count | pct
-----------

## Sampling Manifest

Write `sampling_manifest.json` alongside the sample for reproducibility and Glue migration audit trail.


In [ ]:
# Audit trail: config + quotas + counts for reproducibility and Glue migration.
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "is_glue": IS_GLUE,
    "cell_quotas": quotas,
    "sampled_users": sampled_n,
    "output_counts": output_counts,
    "dummy_output_counts": dummy_output_counts,
    "output_format": {
        "articles": "parquet (unpartitioned)",
        "customers": "parquet (unpartitioned)",
        "transactions": "parquet (hive: year, month)",
    },
    "temporal_split": {
        "cutoff": CONFIG["cutoff_date"],
        "features": "t_dat <= cutoff",
        "stratification": "tier/recency from t_dat <= cutoff only",
        "model_labels": f"t_dat > cutoff within {CONFIG['label_window_days']}-day window (starts day after cutoff)",
        "label_window_days": CONFIG["label_window_days"],
    },
    "notes": {
        "schema": "Column types cast per docs/system-design/schema-info.md (no inferSchema).",
        "images": "dataset/full/images/ intentionally skipped.",
        "dummy": "5 random users from full data, capped at 10 transactions; articles from those txns only.",
        "output_transactions": "All dates retained for sampled users (pre- and post-cutoff).",
    },
}

manifest_path = Path(CONFIG["output_path"]) / "sampling_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Manifest written to {manifest_path}")
